<a href="https://colab.research.google.com/github/myanh-tran/BananaHack-2026---cat-steals-socks/blob/ngu/Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Test

In [8]:
!pip install paddlepaddle

  Using cached paddlepaddle-3.3.1-cp313-cp313-manylinux1_x86_64.whl.metadata (8.8 kB)
Using cached paddlepaddle-3.3.1-cp313-cp313-manylinux1_x86_64.whl (194.8 MB)


In [2]:
# 1. Tải bộ mã nguồn gốc (như đã làm trên Kaggle)
!git clone https://github.com/PaddlePaddle/PaddleOCR.git

# 2. Di chuyển vào thư mục
%cd PaddleOCR

# 3. Chạy lệnh nhận diện bằng script chuẩn của hãng (bỏ qua vỏ bọc bị lỗi)


Cloning into 'PaddleOCR'...
remote: Enumerating objects: 353900, done.
remote: Counting objects: 100% (1212/1212), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 353900 (delta 1126), reused 1038 (delta 1038), pack-reused 352688 (from 2)
Receiving objects: 100% (353900/353900), 1.87 GiB | 19.95 MiB/s, done.
Resolving deltas: 100% (279888/279888), done.
/content/PaddleOCR
/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/content/PaddleOCR/ppocr/postprocess/rec_postprocess.py:1239: SyntaxWarning: invalid escape sequence '\W'
  noletter = "[\W_^\d]"
/content/PaddleOCR/ppocr/postprocess/rec_postprocess.py:1479: SyntaxWarning: invalid escape sequence '\W'
  noletter = "[\W_^\d]"
/content/PaddleOCR/p

In [5]:
!python tools/infer/predict_rec.py \
    --image_dir="/content/oto.png" \
    --rec_model_dir="/content/inference_model" \
    --rec_char_dict_path="/content/dict.txt" \
    --use_gpu=False

/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[2026/10/11 11:03:37] ppocr INFO: In PP-OCRv3, rec_image_shape parameter defaults to '3, 48, 320', if you are using recognition model with PP-OCRv2 or an older version, please set --rec_image_shape='3,32,320
[2026/10/11 11:03:37] ppocr INFO: Predicts of /content/oto.png:('30MF45678', 0.9135012030601501)


In [8]:
!pip install -q paddlepaddle paddleocr ultralytics opencv-python-headless


In [9]:
%cd /content
!ls inference_model

/content
inference.json	inference.pdiparams  inference.yml


In [10]:
import cv2
from plate_ocr import PlateOCR
ocr = PlateOCR("inference_model")
img = cv2.imread("/content/oto.png")      # ảnh biển đã cắt sát
print(ocr.read(img))

('30MF45678', 0.9366124272346497)


In [12]:
from ultralytics import YOLO
det = YOLO("best.pt")
img = cv2.imread("/content/Screenshot 2026-10-10 165719.png")
r = det(img)[0]
for (x1, y1, x2, y2), conf in zip(r.boxes.xyxy.cpu().numpy().astype(int), r.boxes.conf):
    crop = img[y1:y2, x1:x2]
    print(round(float(conf), 2), ocr.read(crop))


0: 256x640 1 License_Plate, 7.9ms
Speed: 1.5ms preprocess, 7.9ms inference, 1.4ms postprocess per image at shape (1, 3, 256, 640)
0.72 ('30M45678', 0.9671866297721863)


In [13]:
from difflib import SequenceMatcher

def match_plate(text, db_plates, min_ratio=0.8, margin=0.1):
    scored = sorted(((SequenceMatcher(None, text, p).ratio(), p) for p in db_plates), reverse=True)
    if not scored or scored[0][0] < min_ratio:
        return None                      # không đủ giống
    if len(scored) > 1 and scored[0][0] - scored[1][0] < margin:
        return None                      # nhập nhằng giữa 2 biển, cần người xác nhận
    return scored[0][1]